# Option A — Pretrained Frozen Embeddings + Classical Classifier
## Strategy
Extract **frozen** visual and text embeddings from a pretrained model (no training).
Use cosine similarity + raw embeddings as features for a classical SVM / Logistic Regression.
No neural network is trained — the pretrained model is used only as a feature extractor.


In [ ]:
# Cell 1 — Imports and setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.utils import shuffle
import warnings
warnings.filterwarnings('ignore')

print("Installing sentence-transformers for text embeddings (pretrained, frozen)...")
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "Pillow", "--quiet"], check=True)

from sentence_transformers import SentenceTransformer
from PIL import Image
import torch
import torchvision.models as tv_models
import torchvision.transforms as transforms

print("All libraries ready.")

In [ ]:
# Cell 2 — Data loading
DATA_DIR = '../data/processed'

def load_split(split_name):
    texts, paths, labels = [], [], []
    for label, cat in enumerate(['incoherent', 'coherent']):
        folder = os.path.join(DATA_DIR, split_name, cat)
        if not os.path.exists(folder):
            print(f"Folder {folder} not found. Skipping.")
            continue
        for f in os.listdir(folder):
            if f.endswith('.txt'):
                txt_path = os.path.join(folder, f)
                with open(txt_path, 'r', encoding='utf-8') as file:
                    texts.append(file.read())
                img_path = os.path.join(folder, f.replace('.txt', '.jpg'))
                paths.append(img_path)
                labels.append(label)  # 0=incoherent, 1=coherent
    return shuffle(texts, paths, np.array(labels), random_state=42)

print("Loading splits...")
t_train, i_train, y_train = load_split('train')
t_val,   i_val,   y_val   = load_split('validation')
t_test,  i_test,  y_test  = load_split('test')

print(f"Train: {len(t_train)} | Val: {len(t_val)} | Test: {len(t_test)}")
print(f"Train class balance: {np.bincount(y_train)}")


In [ ]:
# Cell 3 — Extract frozen text embeddings (sentence-transformers)
# Model is pretrained and FROZEN — we only call .encode(), no training.
print("Loading pretrained text encoder (all-MiniLM-L6-v2)...")
text_model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim, lightweight

def get_text_embeddings(texts, batch_size=64):
    return text_model.encode(texts, batch_size=batch_size, show_progress_bar=True,
                             convert_to_numpy=True, normalize_embeddings=True)

print("Encoding texts...")
E_text_train = get_text_embeddings(t_train)
E_text_val   = get_text_embeddings(t_val)
E_text_test  = get_text_embeddings(t_test)

print(f"Text embedding shape: {E_text_train.shape}")  # (7000, 384)


In [ ]:
# Cell 4 — Extract frozen image embeddings (ResNet-18, pretrained on ImageNet)
# We remove the final classification head — only the feature extractor is used.
print("Loading pretrained image encoder (ResNet-18)...")
resnet = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
# Remove final FC layer → outputs 512-dim feature vector
image_encoder = torch.nn.Sequential(*list(resnet.children())[:-1])
image_encoder.eval()

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

def get_image_embeddings(image_paths, batch_size=64):
    all_feats = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        tensors = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert('RGB')
                tensors.append(preprocess(img))
            except Exception:
                tensors.append(torch.zeros(3, 224, 224))
        batch = torch.stack(tensors)
        with torch.no_grad():
            feats = image_encoder(batch).squeeze(-1).squeeze(-1)  # (B, 512)
        # L2 normalize
        feats = feats / (feats.norm(dim=1, keepdim=True) + 1e-8)
        all_feats.append(feats.numpy())
        if (i // batch_size) % 10 == 0:
            print(f"  Processed {i+len(batch_paths)}/{len(image_paths)}")
    return np.vstack(all_feats)

print("Encoding images (train)...")
E_img_train = get_image_embeddings(i_train)
print("Encoding images (val)...")
E_img_val   = get_image_embeddings(i_val)
print("Encoding images (test)...")
E_img_test  = get_image_embeddings(i_test)

print(f"Image embedding shape: {E_img_train.shape}")  # (7000, 512)


In [ ]:
# Cell 5 — Build cross-modal features
# Key insight: coherence = high similarity between text and image embeddings.
# We give the classifier: [cosine_sim, text_emb, img_emb, elementwise_product]

def build_features(E_text, E_img):
    # 1. Cosine similarity (scalar) — the most important feature
    cos_sim = (E_text * E_img).sum(axis=1, keepdims=True)  # (N, 1)
    
    # 2. Element-wise product — captures feature-level alignment
    product = E_text[:, :min(E_text.shape[1], E_img.shape[1])] *               E_img[:, :min(E_text.shape[1], E_img.shape[1])]  # (N, 384)
    
    # 3. Absolute difference — captures disagreement
    diff = np.abs(E_text[:, :min(E_text.shape[1], E_img.shape[1])] - 
                  E_img[:, :min(E_text.shape[1], E_img.shape[1])])  # (N, 384)
    
    # Concatenate all
    return np.hstack([cos_sim, product, diff])

print("Building cross-modal features...")
X_train = build_features(E_text_train, E_img_train)
X_val   = build_features(E_text_val,   E_img_val)
X_test  = build_features(E_text_test,  E_img_test)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f"Final feature shape: {X_train.shape}")

# Quick check: cosine similarity distribution per class
cos_train = (E_text_train * E_img_train).sum(axis=1)
plt.figure(figsize=(8,4))
plt.hist(cos_train[y_train==0], bins=40, alpha=0.6, label='Incoherent')
plt.hist(cos_train[y_train==1], bins=40, alpha=0.6, label='Coherent')
plt.xlabel('Cosine similarity (text vs image)')
plt.ylabel('Count')
plt.title('Text-Image cosine similarity by class')
plt.legend()
plt.tight_layout()
plt.show()
print("If the two histograms are separated → strong signal!")


In [ ]:
# Cell 6 — Train and compare classical classifiers
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Linear SVM':          LinearSVC(max_iter=3000, C=1.0, random_state=42),
    'RBF SVM':             SVC(kernel='rbf', C=1.0, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    model.fit(X_train, y_train)
    y_val_pred = model.predict(X_val)
    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1  = f1_score(y_val, y_val_pred)
    results[name] = {'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
                     'val_accuracy': val_acc, 'val_f1': val_f1, 'model': model}
    print(f"{name:25s} CV: {cv_scores.mean():.4f} ±{cv_scores.std():.4f} | Val Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

results_df = pd.DataFrame(results).T[['cv_mean','cv_std','val_accuracy','val_f1']]
print("\n=== Comparative summary ===")
print(results_df.sort_values('val_accuracy', ascending=False))


In [ ]:
# Cell 7 — Evaluate best model on test set
best_name = results_df['val_accuracy'].idxmax()
best_model = results[best_name]['model']
print(f"Best model: {best_name}")

y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
test_f1  = f1_score(y_test, y_pred)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1:       {test_f1:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['incoherent','coherent']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['incoherent','coherent'],
            yticklabels=['incoherent','coherent'])
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()


In [ ]:
# Cell 8 — Save everything
import joblib
joblib.dump(best_model, 'optionA_best_model.pkl')
joblib.dump(scaler,     'optionA_scaler.pkl')
print("Saved: optionA_best_model.pkl, optionA_scaler.pkl")
print("\n=== FINAL SUMMARY ===")
print(f"Model : {best_name}")
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test F1       : {test_f1:.4f}")
